**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Digital Communications

How bits become waveforms and survive the trip back: modulation, matched filtering, synchronization, and OFDM — a working QAM link and a working multicarrier link, both built from scratch in NumPy.

## 1. Pre-requisites

- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S4 (matched filters).
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 for the capacity backdrop.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Modulation: Bits → Waveforms* (~35 min)
**Goal:** map bits to constellation symbols; understand the energy/rate trade.
**Feeds into:** Session 2 (matched filter & eyes).

---

## 2. Constellations

💡 **Intuition.** A passband waveform $A\cos(2\pi f_c t + \phi)$ has two independent knobs — amplitude-on-cosine and amplitude-on-sine — so each symbol period carries a **complex number** $I + jQ$. A *constellation* is the alphabet of complex points you allow: QPSK uses 4 (2 bits/symbol), 16-QAM uses 16 (4 bits/symbol). More points ⇒ more bits per symbol ⇒ points closer together ⇒ noise flips them more easily. Modulation design is packing points on a power budget — with [capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) as the referee.

In [2]:
def qam16():
    pts = np.array([a + 1j*b for a in (-3,-1,1,3) for b in (-3,-1,1,3)])
    return pts / np.sqrt((np.abs(pts)**2).mean())          # unit average energy

def qpsk():
    return np.exp(1j * (np.pi/4 + np.pi/2 * np.arange(4)))

n_sym = 3000
for name, const, snr_db in [("QPSK", qpsk(), 10), ("16-QAM", qam16(), 10)]:
    tx = const[rng.integers(0, len(const), n_sym)]
    noise = (rng.standard_normal(n_sym) + 1j*rng.standard_normal(n_sym)) / np.sqrt(2)
    rx = tx + noise * 10**(-snr_db/20)
    plt.scatter(rx.real, rx.imag, s=2, alpha=0.4, label=f"{name} @ {snr_db} dB")
plt.axis("equal"); plt.legend(); plt.title("Same SNR, two alphabets: density costs margin")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2038435/2927047130.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *Pulse Shaping, Matched Filters & Eye Diagrams* (~40 min)
**Goal:** put symbols on pulses without smearing neighbors; read link health from the eye.
**Builds on:** Session 1; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 3 (synchronization).

---

## 3. From Symbols to Samples

💡 **Intuition.** You can't transmit points — you transmit *pulses* carrying points, and bandwidth limits force pulses to be long, overlapping their neighbors. The escape is **Nyquist pulses** (raised cosine): they may overlap everywhere *except at the sampling instants*, where every other pulse crosses zero — no inter-symbol interference *if you sample exactly on time*. The receiver applies the [matched filter](./Statistical_Signal_Processing.ipynb) for SNR, and the **eye diagram** — all symbol periods overlaid — shows your margins: vertical opening = noise margin, horizontal = timing margin.

In [3]:
# Root-raised-cosine link, oversampled 8x
sps, beta, span = 8, 0.35, 8
t_rrc = np.arange(-span*sps, span*sps + 1) / sps
def rrc(t, beta):
    out = np.zeros_like(t, float)
    for i, ti in enumerate(t):
        if abs(ti) < 1e-9: out[i] = 1 - beta + 4*beta/np.pi
        elif abs(abs(4*beta*ti) - 1) < 1e-9:
            out[i] = beta/np.sqrt(2)*((1+2/np.pi)*np.sin(np.pi/(4*beta)) + (1-2/np.pi)*np.cos(np.pi/(4*beta)))
        else:
            out[i] = (np.sin(np.pi*ti*(1-beta)) + 4*beta*ti*np.cos(np.pi*ti*(1+beta))) / (np.pi*ti*(1-(4*beta*ti)**2))
    return out / np.sqrt((out**2).sum())
h_rrc = rrc(t_rrc, beta)

syms = qpsk()[rng.integers(0, 4, 500)]
up = np.zeros(len(syms)*sps, complex); up[::sps] = syms
tx_wave = np.convolve(up, h_rrc)

snr_db = 14
noise = (rng.standard_normal(len(tx_wave)) + 1j*rng.standard_normal(len(tx_wave))) / np.sqrt(2)
rx_wave = tx_wave + noise * 10**(-snr_db/20) / np.sqrt(sps)
mf_out = np.convolve(rx_wave, h_rrc)                      # matched filter (RRC ∗ RRC = raised cosine)

delay = len(h_rrc) - 1
plt.figure(figsize=(8, 3))
for k in range(60, 260):                                   # overlay 2-symbol windows: the EYE
    seg = mf_out[delay + k*sps - sps : delay + k*sps + sps]
    plt.plot(np.arange(-sps, sps)/sps, seg.real, "C0", alpha=0.12, linewidth=0.8)
plt.axvline(0, color="r", linestyle=":", linewidth=1)
plt.title("Eye diagram @ 14 dB: open eye = healthy link; sample at the red line")
plt.xlabel("time [symbols]"); plt.tight_layout(); plt.show()

rx_syms = mf_out[delay::sps][:len(syms)]
decided = qpsk()[np.argmin(np.abs(rx_syms[:, None] - qpsk()[None, :]), axis=1)]
print(f"symbol error rate at 14 dB: {(decided != syms).mean():.4f} over {len(syms)} symbols")

symbol error rate at 14 dB: 0.0000 over 500 symbols


/tmp/ipykernel_2038435/1375707096.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("time [symbols]"); plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *Synchronization* (~35 min)
**Goal:** find the frame and the phase: correlation sync and the cost of being wrong.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (OFDM).

---

## 4. Where Does the Frame Start?

💡 **Intuition.** The receiver knows neither *when* symbols start nor *what phase* the oscillator drifted to. Both are solved with correlation: prepend a known **preamble**; the receiver slides it across the incoming stream, and the correlation peak marks the frame start ([matched filter](./Statistical_Signal_Processing.ipynb) again — detection in time). The *phase* of that same peak reveals the carrier phase offset — one correlation, two syncs. Every Wi-Fi packet begins exactly this way.

In [4]:
preamble = qpsk()[rng.integers(0, 4, 64)]
payload = qpsk()[rng.integers(0, 4, 400)]
frame = np.concatenate([preamble, payload])

phase_off = 0.6                                          # unknown carrier phase [rad]
start = 137                                              # unknown start position
stream = np.concatenate([ (rng.standard_normal(start)+1j*rng.standard_normal(start))*0.2,
                          frame * np.exp(1j*phase_off),
                          (rng.standard_normal(200)+1j*rng.standard_normal(200))*0.2 ])
stream += (rng.standard_normal(len(stream)) + 1j*rng.standard_normal(len(stream))) * 0.1

corr = np.abs(np.correlate(stream, preamble, "valid"))
est_start = int(np.argmax(corr))
peak = np.correlate(stream, preamble, "valid")[est_start]
est_phase = np.angle(peak / (np.abs(preamble)**2).sum() * len(preamble)) if False else np.angle(peak)

plt.figure(figsize=(8, 2.4))
plt.plot(corr); plt.axvline(start, color="r", linestyle=":", label=f"true start {start}")
plt.legend(); plt.title(f"preamble correlation: peak at {est_start}, phase estimate {est_phase:.3f} rad (true {phase_off})")
plt.tight_layout(); plt.show()

fixed = stream[est_start + 64 : est_start + 64 + 400] * np.exp(-1j * est_phase)
decided = qpsk()[np.argmin(np.abs(fixed[:, None] - qpsk()[None, :]), axis=1)]
print(f"payload symbol errors after sync: {(decided != payload).sum()} / 400")

payload symbol errors after sync: 0 / 400


/tmp/ipykernel_2038435/234182044.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *OFDM in 40 Minutes* (~40 min)
**Goal:** beat multipath by going wide-and-slow: the FFT as a modem.
**Builds on:** Session 3; [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S7.

---

## 5. OFDM

💡 **Intuition.** Multipath (echoes) smears fast single-carrier symbols into each other. OFDM's judo move: send **many slow streams in parallel**, one per subcarrier — and use the IFFT to pack them, the FFT to unpack. The **cyclic prefix** turns the channel's *linear* convolution into *circular* convolution ([Foundations 1 S8](./Foundations_of_Signal_Processing_1.ipynb)!), so the whole channel collapses to one complex multiply per subcarrier — equalization becomes division. Wi-Fi, LTE/5G, and DVB are exactly this.

In [5]:
Nfft, cp, n_ofdm = 64, 16, 200
channel = np.array([1.0, 0, 0.5, 0, 0, 0.3j])            # nasty multipath

data = qpsk()[rng.integers(0, 4, (n_ofdm, Nfft))]
tx = []
for row in data:
    sym = np.fft.ifft(row) * np.sqrt(Nfft)
    tx.append(np.concatenate([sym[-cp:], sym]))           # cyclic prefix
tx = np.concatenate(tx)

rx = np.convolve(tx, channel)[:len(tx)]
rx += (rng.standard_normal(len(rx)) + 1j*rng.standard_normal(len(rx))) * 0.05

H = np.fft.fft(channel, Nfft)                             # channel per subcarrier (known/est. via pilots)
eq_syms, raw_syms = [], []
for k in range(n_ofdm):
    blk = rx[k*(Nfft+cp)+cp : (k+1)*(Nfft+cp)]
    Y = np.fft.fft(blk) / np.sqrt(Nfft)
    raw_syms.append(Y); eq_syms.append(Y / H)             # equalization = one division!
raw = np.concatenate(raw_syms); eq = np.concatenate(eq_syms)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].scatter(raw.real, raw.imag, s=1, alpha=0.3); axes[0].set_title("before equalization: smeared by multipath")
axes[1].scatter(eq.real, eq.imag, s=1, alpha=0.3); axes[1].set_title("after Y/H: constellation restored")
for ax in axes: ax.set_aspect("equal")
plt.tight_layout(); plt.show()

decided = qpsk()[np.argmin(np.abs(eq.ravel()[:, None] - qpsk()[None, :]), axis=1)]
print(f"OFDM symbol error rate through multipath: {(decided != data.ravel()).mean():.4f}")

OFDM symbol error rate through multipath: 0.0002


/tmp/ipykernel_2038435/1503258650.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

Bits ride complex symbols; Nyquist pulses dodge ISI; one correlation finds both time and phase; and OFDM turns a hostile channel into $N$ trivial ones via the FFT. You've built every layer of a real modem below the error-correcting code.

---
## Where next

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the capacity these designs chase.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — run this against *real* airwaves.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate front-ends around every modem.